# 程式來源核對與小型修正消融
不等同原始V5/V11重現，也沒有通過驗證的新提交。此notebook未在Colab平台執行。私有GitHub repository需安全授權，不要將token寫進檔案。

In [ ]:
from pathlib import Path
import os,sys,subprocess,tempfile,json
repo=Path("/content/text")
if not repo.exists():
    subprocess.run(["git","clone","--branch","research-0680-20261024","https://github.com/yeyu0302/text.git",str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,"-m","pip","install","-r","research_0680/configs/requirements.txt"],check=True)
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",MPLCONFIGDIR="/tmp/phase8-mpl")


In [ ]:
audit=Path(tempfile.mkdtemp(prefix="phase8-audit-"))
env["RESEARCH0680_PHASE8_AUDIT_RESULTS"]=str(audit)
subprocess.run([sys.executable,"-m","research_0680.phase8.source_audit"],env=env,check=True)
print(json.loads((audit/"source_findings.json").read_text()))


下格重現已完成的小型數值修訂比較，約數秒至數十秒。它只用holdout，不是新OOF。也可直接閱讀repository已保存結果，無需反覆重跑。

In [ ]:
output=Path(tempfile.mkdtemp(prefix="phase8-probe-"))
env["RESEARCH0680_PHASE8_RESULTS"]=str(output)
env["RESEARCH0680_PHASE8_PROTOCOL"]=str(repo/"research_0680/phase8/convergence_protocol.json")
subprocess.run([sys.executable,"-m","research_0680.phase8.ablation"],env=env,check=True)
import pandas as pd
display(pd.DataFrame(json.loads((output/"screen.json").read_text())["metrics"]).T)


In [ ]:
subprocess.run([sys.executable,"-m","pytest","-q","research_0680/phase8/tests"],env=env,check=True)
